
# Air Traffic Data Analysis — Exercises XP (Solution Notebook)

**Author:** _Your Name (Ariel Koss?)_  
**Date:** _Automatically generated_

This notebook completes the full workflow requested in the exercise:
1. Setup & Data Loading  
2. Exploratory Data Analysis (EDA)  
3. Hypothesis Testing  
4. Simple Linear Regression  
5. Multiple Linear Regression  
6. Model Comparison & Analysis  
7. Statistical Insights & Conclusions  
8. Reflection Questions

> Notes
> - Charts are built with **matplotlib** only (no seaborn).
> - If `air_traffic_data.csv` is missing, the notebook will generate a realistic synthetic dataset.
> - You can run this end-to-end. Each section is independent and commented for clarity.



## Section 1 — Setup & Data Loading

**What we do:**
- Import libraries
- Load `air_traffic_data.csv` if present, otherwise generate synthetic data with realistic correlations
- Quick peek at shape and basic info


In [ ]:

# === Imports ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Stats & ML
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Display options
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

DATA_PATH = 'air_traffic_data.csv'

# === Load or generate data ===
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    # ---- Synthetic data generator (reproducible, realistic) ----
    rng = np.random.default_rng(42)
    n = 240  # ~20 years monthly
    
    # Base latent demand trend + seasonality for passengers
    months = np.arange(n)
    season = 0.15 * np.sin(2 * np.pi * months / 12.0)  # yearly seasonality
    trend = 1.0 + 0.002 * months                        # slow upward trend
    
    # Domestic / International split with noise
    base_dom = rng.normal(60_000, 8_000, size=n) * trend * (1 + season*0.6)
    base_int = rng.normal(55_000, 9_500, size=n) * trend * (1 + season*0.9)
    Dom_Pax = np.maximum(5_000, base_dom).round().astype(int)
    Int_Pax = np.maximum(5_000, base_int).round().astype(int)
    Pax = (Dom_Pax + Int_Pax).astype(int)
    
    # Flights scaling with passengers plus noise
    Dom_Flt = np.maximum(500, (Dom_Pax / 120).astype(int) + rng.integers(-50, 50, size=n))
    Int_Flt = np.maximum(450, (Int_Pax / 140).astype(int) + rng.integers(-50, 60, size=n))
    Flt = Dom_Flt + Int_Flt
    
    # Domestic RPM roughly proportional to Dom_Pax and average stage length
    avg_stage_len_dom = rng.normal(550, 60, size=n)  # miles
    Dom_RPM = (Dom_Pax * avg_stage_len_dom).astype(int)
    
    df = pd.DataFrame({
        'Dom_Pax': Dom_Pax,
        'Int_Pax': Int_Pax,
        'Pax': Pax,
        'Dom_Flt': Dom_Flt,
        'Int_Flt': Int_Flt,
        'Flt': Flt,
        'Dom_RPM': Dom_RPM,
    })
    
    # Save a copy for reproducibility
    df.to_csv(DATA_PATH, index=False)

print('Shape:', df.shape)
df.head(10)



## Section 2 — Exploratory Data Analysis (EDA)

**Tasks:**
- `df.info()`, `df.head()`, `df.describe()`
- Check missing values
- Correlation matrix & heatmap (matplotlib)
- Identify strongest correlations


In [ ]:

# Basic info
display(df.info())
display(df.describe().T)

# Missing values
print('\nMissing values per column:')
print(df.isnull().sum())

# Correlation matrix
corr = df.corr(numeric_only=True)

# Heatmap with matplotlib (no seaborn)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, interpolation='nearest')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Annotate cells
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)

ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Identify strongest absolute correlations (excluding self-correlation)
corr_unstack = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
top_pairs = corr_unstack.sort_values(ascending=False).head(10)
print('\nTop correlation pairs (absolute):')
display(top_pairs)



## Section 3 — Hypothesis Testing

**Test 1:** Compare mean Domestic vs International Passengers  
- H₀: μ_dom = μ_int  
- H₁: μ_dom ≠ μ_int  
Method: independent samples t-test

**Test 2:** Correlation significance between Total Passengers (`Pax`) and Total Flights (`Flt`)  
- H₀: ρ = 0  
- H₁: ρ ≠ 0  
Method: Pearson correlation test


In [ ]:

alpha = 0.05

# Test 1: t-test (independent)
t_stat, p_val = stats.ttest_ind(df['Dom_Pax'], df['Int_Pax'], equal_var=False)  # Welch's t-test safer
print('Test 1 — Dom_Pax vs Int_Pax')
print(f't-statistic: {t_stat:.3f}, p-value: {p_val:.6f}')
if p_val < alpha:
    print('Result: Reject H0 at α=0.05 — means differ significantly.')
else:
    print('Result: Fail to reject H0 at α=0.05 — no significant difference detected.')

# Test 2: Pearson correlation Pax vs Flt
r, p = stats.pearsonr(df['Pax'], df['Flt'])
print('\nTest 2 — Correlation (Pax vs Flt)')
print(f'Pearson r: {r:.3f}, p-value: {p:.6f}')
if p < alpha:
    print('Result: Reject H0 — significant linear correlation between Pax and Flt.')
else:
    print('Result: Fail to reject H0 — no significant linear correlation detected.')



## Section 4 — Simple Linear Regression

**Goal:** Predict total passengers `Pax` using total flights `Flt`.  
We build a `LinearRegression`, evaluate on a hold-out test set, and visualize fit + residuals.


In [ ]:

# Features and target
X = df[['Flt']].values
y = df['Pax'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
linr_simple = LinearRegression()
linr_simple.fit(X_train, y_train)

# Predict
y_pred = linr_simple.predict(X_test)

# Metrics
r2_simple = r2_score(y_test, y_pred)
mse_simple = mean_squared_error(y_test, y_pred)
rmse_simple = np.sqrt(mse_simple)
mae_simple = mean_absolute_error(y_test, y_pred)

print('Simple Linear Regression (Pax ~ Flt)')
print(f'Intercept: {linr_simple.intercept_:,.3f}')
print(f'Coefficient (Flt): {linr_simple.coef_[0]:,.3f}')
print(f'R^2: {r2_simple:.4f}')
print(f'MSE: {mse_simple:,.2f}')
print(f'RMSE: {rmse_simple:,.2f}')
print(f'MAE: {mae_simple:,.2f}')

# Visualization: scatter + regression line
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(X_test, y_test, alpha=0.6, label='Test data')
# Line
grid = np.linspace(X_test.min(), X_test.max(), 100).reshape(-1,1)
ax.plot(grid, linr_simple.predict(grid), linewidth=2, label='Fitted line')
ax.set_xlabel('Flt (Total Flights)')
ax.set_ylabel('Pax (Total Passengers)')
ax.set_title('Simple Linear Regression — Test Set')
ax.legend()
plt.tight_layout()
plt.show()

# Residuals
residuals = y_test - y_pred

# Residuals vs Fitted
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(y_pred, residuals, alpha=0.6)
ax.axhline(0, linestyle='--')
ax.set_xlabel('Fitted values (ŷ)')
ax.set_ylabel('Residuals (y - ŷ)')
ax.set_title('Residuals vs Fitted (Simple LR)')
plt.tight_layout()
plt.show()

# QQ-plot (approx) using scipy probplot + matplotlib
fig = plt.figure(figsize=(6,6))
_ = stats.probplot(residuals, dist='norm', plot=plt)
plt.title('QQ-Plot of Residuals (Simple LR)')
plt.tight_layout()
plt.show()



## Section 5 — Multiple Linear Regression

**Goal:** Predict `Pax` from multiple predictors while avoiding leakage/multicollinearity.  
**Features used:** `Dom_Pax`, `Int_Pax`, `Dom_Flt`, `Int_Flt`, `Dom_RPM`  
(We exclude `Pax` and `Flt` since they are sums involving target-related components.)


In [ ]:

features = ['Dom_Pax', 'Int_Pax', 'Dom_Flt', 'Int_Flt', 'Dom_RPM']
target = 'Pax'

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (fit on train, transform train & test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

linr_multi = LinearRegression()
linr_multi.fit(X_train_scaled, y_train)

y_pred_multi = linr_multi.predict(X_test_scaled)

r2_multi = r2_score(y_test, y_pred_multi)
mse_multi = mean_squared_error(y_test, y_pred_multi)
rmse_multi = np.sqrt(mse_multi)
mae_multi = mean_absolute_error(y_test, y_pred_multi)

print('Multiple Linear Regression')
print('Features:', features)
print('Coefficients (scaled features):')
for name, coef in zip(features, linr_multi.coef_):
    print(f'  {name:>10s}: {coef:,.4f}')
print(f'Intercept: {linr_multi.intercept_:,.3f}')
print(f'R^2: {r2_multi:.4f}')
print(f'MSE: {mse_multi:,.2f}')
print(f'RMSE: {rmse_multi:,.2f}')
print(f'MAE: {mae_multi:,.2f}')



## Section 6 — Model Comparison & Analysis

We compare `Simple LR (Pax ~ Flt)` vs `Multiple LR` on R², RMSE, and MAE and compute percent improvements.


In [ ]:

def improvement_pct(old, new, higher_is_better):
    if higher_is_better:
        # e.g., R^2
        return 100.0 * (new - old) / (abs(old) + 1e-12)
    else:
        # e.g., errors where lower is better
        return 100.0 * (old - new) / (abs(old) + 1e-12)

comparison = pd.DataFrame({
    'Metric': ['R2', 'RMSE', 'MAE'],
    'Simple_LR': [r2_simple, rmse_simple, mae_simple],
    'Multiple_LR': [r2_multi, rmse_multi, mae_multi],
})

# Add improvement col (Multiple vs Simple)
comparison['Improvement_%'] = [
    improvement_pct(r2_simple, r2_multi, higher_is_better=True),
    improvement_pct(rmse_simple, rmse_multi, higher_is_better=False),
    improvement_pct(mae_simple, mae_multi, higher_is_better=False),
]

display(comparison)



## Section 7 — Statistical Insights & Conclusions

This section prints a concise, business-oriented summary based on the results so far.


In [ ]:

print('--- Insights & Conclusions ---')
print('* Hypothesis Test 1 (Dom vs Int Pax):')
print('  -> See t-test results above for significance. If p<0.05, the mean levels differ; plan capacity accordingly.')

print('\n* Hypothesis Test 2 (Pax vs Flt correlation):')
print('  -> If significant and positive, more flights align with higher passenger volume; flight scheduling can closely track demand.')

print('\n* Simple LR (Pax ~ Flt):')
print(f'  -> R^2={r2_simple:.3f}. Good fit implies flights explain a substantial share of passenger variance.')
print('  -> Residual checks: look for randomness around 0; patterns suggest nonlinearity or omitted variables.')

print('\n* Multiple LR:')
print(f'  -> R^2={r2_multi:.3f}. Typically higher than simple LR when informative features are added.')
print('  -> Scaled coefficients indicate relative importance; large magnitudes suggest stronger association.')

print('\n* Recommendation examples:')
print('  1) Use flight counts and domestic/international splits to forecast monthly demand more accurately.')
print('  2) If international demand drives variance, reallocate widebodies/slots seasonally.')
print('  3) Track RPM and stage length to optimize network planning (yields, route economics).')
print('  4) Monitor residuals; if heteroscedastic, consider transformations or robust regressors.')



## Section 8 — Reflection Questions (Short Answers)

**Q1.** What do hypothesis test results reveal about air traffic patterns?  
*Answer template:* If means differ (p<0.05), domestic and international demand have distinct levels or seasonality impacts, guiding differentiated capacity planning.

**Q2.** Why did one regression model perform better than the other?  
*Answer template:* Multiple regression captures more signal (e.g., Dom/Int splits, RPM), reducing omitted-variable bias vs. a single predictor.

**Q3.** How can airlines use correlation insights operationally?  
*Answer template:* Strong Pax–Flt correlation helps align scheduling with demand; correlations with RPM inform route optimization and aircraft assignment.

**Q4.** What do residual plots tell you about model assumptions?  
*Answer template:* Random scatter supports linearity and homoscedasticity; patterns imply nonlinearity, missing features, or variance issues.

**Q5.** What are practical applications of these models?  
*Answer template:* Monthly demand forecasting, staffing/slot planning, fleet assignment, and revenue management baselines.
